In [ ]:
# Rebuilt one-click Colab (max identity defaults)
# Stable startup: prints Gradio URL and Colab-proxy fallback URL directly.

!pip install -q pygit2==1.15.1
!pip install -q "numpy<2"

import os
import re
import shutil
import subprocess
import sys
from IPython.display import HTML, display

os.chdir('/content')
if os.path.exists('/content/Fooocus'):
    shutil.rmtree('/content/Fooocus')

REPO_URL = "https://github.com/sunsideaspect/foocus_new.git"
BRANCH = "cursor/git-eaf0"  # switch to "main" after merge if needed
subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, "Fooocus"], check=True)

os.chdir('/content/Fooocus')
cmd = [
    sys.executable,
    "entry_with_update.py",
    "--preset", "realistic_identity",
    "--share",
    "--always-high-vram",  # replace with --always-normal-vram for lower VRAM
    "--disable-in-browser"
]
print("Launching:", " ".join(cmd))

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

share_re = re.compile(r"https://[a-z0-9-]+\.gradio\.live")
local_re = re.compile(r"Running on local URL:\s+(http://127\.0\.0\.1:(\d+))")
printed_links = False

for line in proc.stdout:
    print(line, end="")

    m = share_re.search(line)
    if m and not printed_links:
        share_url = m.group(0)
        print("\n✅ Gradio public URL:\n" + share_url + "\n")
        display(HTML(f'<a href="{share_url}" target="_blank">Open Gradio public URL</a>'))
        printed_links = True

    lm = local_re.search(line)
    if lm:
        port = lm.group(2)
        try:
            from google.colab import output
            proxy_url = output.eval_js(f"google.colab.kernel.proxyPort({port})")
            if proxy_url:
                print("\n✅ Colab proxy fallback URL:\n" + proxy_url + "\n")
                display(HTML(f'<a href="{proxy_url}" target="_blank">Open Colab proxy URL (fallback)</a>'))
        except Exception as e:
            print(f"[Info] Could not create Colab proxy URL: {e}")
